# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [71]:
import pandas as pd

In [73]:
data = pd.read_csv('previsao_de_renda.csv')

In [75]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [77]:
from sklearn.model_selection import train_test_split

# Separar os dados em features (X) e target (y)
X = data.drop(columns=['renda', 'Unnamed: 0', 'id_cliente'])
y = data['renda']

# Dividir a base em treinamento (75%) e teste (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Mostrar as dimensões das bases
(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


((11250, 12), (3750, 12), (11250,), (3750,))

In [21]:
#2
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Separar colunas numéricas e categóricas
numeric_features = ['idade', 'tempo_emprego', 'qtd_filhos', 'qt_pessoas_residencia']
categorical_features = [
    'sexo', 'posse_de_veiculo', 'posse_de_imovel', 'tipo_renda', 
    'educacao', 'estado_civil', 'tipo_residencia'
]

# Atualizar o pré-processador para lidar com valores ausentes
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='first'))
        ]), categorical_features)
    ]
)

# Lista de valores de alpha
alphas = [0, 0.001, 0.005, 0.01, 0.05, 0.1]

# Avaliar modelos Ridge com diferentes valores de alpha após tratar os valores ausentes
results = []
for alpha in alphas:
    ridge = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Ridge(alpha=alpha))
    ])
    ridge.fit(X_train, y_train)
    y_pred = ridge.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    results.append((alpha, r2))

# Converter resultados para DataFrame e exibir o melhor modelo
results_df = pd.DataFrame(results, columns=['alpha', 'r2_score'])
best_model = results_df.loc[results_df['r2_score'].idxmax()]

results_df, best_model


(   alpha  r2_score
 0  0.000  0.269116
 1  0.001  0.269116
 2  0.005  0.269116
 3  0.010  0.269116
 4  0.050  0.269116
 5  0.100  0.269117,
 alpha       0.100000
 r2_score    0.269117
 Name: 5, dtype: float64)

Os resultados mostram o desempenho do modelo Ridge com diferentes valores de alpha:

O melhor modelo foi obtido com alpha = 0.1, alcançando um R² = 0.2691 na base de teste.
Os valores de R² são consistentes para diferentes alphas, mas ligeiramente melhores com alpha maior.

In [25]:
# Ajustar o processo para evitar erros no LASSO
results_lasso_corrected = []

for alpha in alphas[1:]:  # Exclui alpha = 0, que não é adequado para LASSO
    lasso = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Lasso(alpha=alpha, max_iter=10000))
    ])
    lasso.fit(X_train, y_train)
    y_pred = lasso.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    results_lasso_corrected.append((alpha, r2))

# Converter resultados para DataFrame e identificar o melhor modelo
results_lasso_df_corrected = pd.DataFrame(results_lasso_corrected, columns=['alpha', 'r2_score'])
best_lasso_model_corrected = results_lasso_df_corrected.loc[results_lasso_df_corrected['r2_score'].idxmax()]

results_lasso_df_corrected, best_lasso_model_corrected


(   alpha  r2_score
 0  0.001  0.269116
 1  0.005  0.269116
 2  0.010  0.269117
 3  0.050  0.269120
 4  0.100  0.269123,
 alpha       0.100000
 r2_score    0.269123
 Name: 4, dtype: float64)

O método Lasso chegou a um melhor resultado que o modelo Ridge

In [91]:
# Codificar variáveis categóricas
def encode_categorical_features(df):
    label_encoders = {}
    for column in df.select_dtypes(include=['object', 'bool']):
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column].astype(str))
        label_encoders[column] = le
    return df, label_encoders

X_train_encoded, encoders = encode_categorical_features(X_train.copy())
X_test_encoded, _ = encode_categorical_features(X_test.copy())

# Tratar valores ausentes
X_train_encoded = X_train_encoded.fillna(X_train_encoded.median(numeric_only=True))
X_test_encoded = X_test_encoded.fillna(X_train_encoded.median(numeric_only=True))

# Stepwise Selection
def stepwise_selection(X, y, initial_list=[], threshold_in=0.01, threshold_out=0.05):
    included = list(initial_list)
    while True:
        changed = False
        # Forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True

        # Backward step
        model = sm.OLS(y, sm.add_constant(X[included])).fit()
        pvalues = model.pvalues.iloc[1:]  # Excluir a constante
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)

        if not changed:
            break
    return included

# Rodar seleção stepwise
selected_features = stepwise_selection(X_train_encoded, y_train)

# Reavaliar com as variáveis selecionadas
X_train_stepwise = X_train_encoded[selected_features]
X_test_stepwise = X_test_encoded[selected_features]

# Adicionar constante para os conjuntos de treino e teste
X_train_stepwise = sm.add_constant(X_train_stepwise)
X_test_stepwise = sm.add_constant(X_test_stepwise, has_constant='add')

# Treinar o modelo
model = sm.OLS(y_train, X_train_stepwise).fit()

# Avaliar o desempenho
y_pred = model.predict(X_test_stepwise)
r2_test = r2_score(y_test, y_pred)

# Exibir os resultados
print(model.summary())
print(f"R² na base de teste: {r2_test:.4f}")

                            OLS Regression Results                            
Dep. Variable:                  renda   R-squared:                       0.255
Model:                            OLS   Adj. R-squared:                  0.255
Method:                 Least Squares   F-statistic:                     961.7
Date:                Sun, 01 Dec 2024   Prob (F-statistic):               0.00
Time:                        18:24:14   Log-Likelihood:            -1.1581e+05
No. Observations:               11250   AIC:                         2.316e+05
Df Residuals:                   11245   BIC:                         2.317e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const         -2023.9252    434.388     -4.659

In [93]:
print(model.params)

const           -2023.925205
tempo_emprego     556.868966
sexo             5856.636501
idade              20.068558
educacao          311.586457
dtype: float64


Resumo Comparativo dos Modelos: Ridge, LASSO e Stepwise

Ridge Regression:

Prós: Reduz overfitting e mantém todas as variáveis.
Contras: Menos interpretável devido à falta de seleção de variáveis.
Desempenho: Equilíbrio consistente entre treino e teste.

LASSO Regression:

Prós: Elimina variáveis irrelevantes, criando um modelo mais simples e interpretável.
Contras: Sensível ao 𝛼.

Desempenho: Competitivo no treino e teste, com boa generalização.

Stepwise Selection:

Prós: Interpretação direta, focando em variáveis estatisticamente relevantes.
Contras: Maior R² no treino, mas queda no teste (potencial overfitting).
Desempenho: Melhor no treino, mas com generalização mais limitada.

Recomendação:
LASSO é o melhor modelo para este caso devido ao equilíbrio entre desempenho e simplicidade, sendo ideal para interpretar variáveis relevantes.

In [105]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
import numpy as np

warnings.filterwarnings("ignore", category=ConvergenceWarning)
# Criar novas variáveis no conjunto de treino e teste
X_train_transformed = X_train_encoded.copy()
X_test_transformed = X_test_encoded.copy()

# 1. Adicionar interações entre variáveis
X_train_transformed['tempo_emprego_idade'] = X_train_transformed['tempo_emprego'] * X_train_transformed['idade']
X_test_transformed['tempo_emprego_idade'] = X_test_transformed['tempo_emprego'] * X_test_transformed['idade']

# 2. Adicionar quadrados de variáveis contínuas
for column in ['tempo_emprego', 'idade']:
    X_train_transformed[f'{column}_squared'] = X_train_transformed[column] ** 2
    X_test_transformed[f'{column}_squared'] = X_test_transformed[column] ** 2

# 3. Adicionar transformações logarítmicas
for column in ['tempo_emprego', 'idade']:
    X_train_transformed[f'log_{column}'] = (X_train_transformed[column] + 1).apply(np.log)
    X_test_transformed[f'log_{column}'] = (X_test_transformed[column] + 1).apply(np.log)

# Ajustar Ridge
ridge_search = GridSearchCV(Pipeline([
    ('scaler', StandardScaler()), 
    ('model', Ridge())
]), param_grid={'model__alpha': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='r2')

ridge_search.fit(X_train_transformed, y_train)
ridge_best = ridge_search.best_estimator_
ridge_r2_test = r2_score(y_test, ridge_best.predict(X_test_transformed))

# Ajustar LASSO
lasso_search = GridSearchCV(Pipeline([
    ('scaler', StandardScaler()), 
    ('model', Lasso(max_iter=10000))
]), param_grid={'model__alpha': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='r2')

lasso_search.fit(X_train_transformed, y_train)
lasso_best = lasso_search.best_estimator_
lasso_r2_test = r2_score(y_test, lasso_best.predict(X_test_transformed))

# Resultados finais
{
    'Ridge Best Alpha': ridge_search.best_params_['model__alpha'],
    'Ridge R2 Test': ridge_r2_test,
    'LASSO Best Alpha': lasso_search.best_params_['model__alpha'],
    'LASSO R2 Test': lasso_r2_test
}


{'Ridge Best Alpha': 100,
 'Ridge R2 Test': 0.2637276350682406,
 'LASSO Best Alpha': 10,
 'LASSO R2 Test': 0.2632143256439792}

In [97]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Ajustar uma árvore de regressão
tree_model = DecisionTreeRegressor(random_state=42)

# GridSearch para encontrar a profundidade ideal da árvore
param_grid_tree = {'max_depth': [3, 5, 7, 10, 15, None]}
tree_search = GridSearchCV(tree_model, param_grid=param_grid_tree, cv=5, scoring='r2')
tree_search.fit(X_train_transformed, y_train)

# Melhor modelo encontrado
best_tree_model = tree_search.best_estimator_

# Previsões e cálculo do R^2
y_pred_tree = best_tree_model.predict(X_test_transformed)
r2_tree_test = r2_score(y_test, y_pred_tree)

# Resultados
print({
    'Best Tree Max Depth': tree_search.best_params_['max_depth'],
    'Tree R2 Test': r2_tree_test
})


{'Best Tree Max Depth': 5, 'Tree R2 Test': 0.39435123344840006}
